In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/26 04:01:00 WARN Utils: Your hostname, fais resolves to a loopback address: 127.0.1.1; using 10.198.113.43 instead (on interface wlp3s0)
26/10/26 04:01:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 04:01:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
!start-dfs.sh
!start-yarn.sh

Starting namenodes on [localhost]
localhost: namenode is running as process 5813.  Stop it first and ensure /tmp/hadoop-fais-namenode.pid file is empty before retry.
Starting datanodes
localhost: datanode is running as process 5976.  Stop it first and ensure /tmp/hadoop-fais-datanode.pid file is empty before retry.
Starting secondary namenodes [fais]
fais: secondarynamenode is running as process 6253.  Stop it first and ensure /tmp/hadoop-fais-secondarynamenode.pid file is empty before retry.
Starting resourcemanager
Starting nodemanagers
localhost: nodemanager is running as process 6636.  Stop it first and ensure /tmp/hadoop-fais-nodemanager.pid file is empty before retry.


## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dari awal modul, lalu baca kembali `data_transaksi_besar.csv` dan `data_pelanggan.json` sebelum mengerjakan latihan berikut.

In [3]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Baca kembali `data_transaksi_besar_parquet` (hasil Sub-bab 6.1.1) menggunakan `spark.read.parquet()`, lalu tampilkan `printSchema()`-nya. Bandingkan dengan skema hasil `inferSchema=True` pada CSV — apakah tipe datanya sama?

In [4]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
csv_size = os.path.getsize("data_transaksi_besar.csv")
df_transaksi_etl.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

# Membaca kembali Parquet — perhatikan kita TIDAK perlu inferSchema=True
# karena skema/tipe data sudah tersimpan built-in di dalam berkas Parquet itu sendiri
df_dari_parquet = spark.read.parquet("data_transaksi_besar_parquet")
df_dari_parquet.printSchema()
df_dari_parquet.show(5)

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------

**Soal 2.** Gabungkan `df_transaksi_etl` dengan `df_pelanggan_etl`, lalu tampilkan jumlah transaksi untuk masing-masing nilai `membership` (`Silver`/`Gold`/`Platinum`).

In [5]:
print("Data Transaksi:")
df_transaksi_etl.show(5)
print("Data Pelanggan:")
df_pelanggan_etl.show(5)
# Gabungkan df_transaksi_etl dengan df_pelanggan_etl berdasarkan kolom pengikat (misal: customer_id)
df_gabung = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="inner")

# Kelompokkan berdasarkan kolom 'membership' dan hitung jumlah transaksinya
df_hasil = df_gabung.groupBy("membership").count().withColumnRenamed("count", "jumlah_transaksi")

# Tampilkan hasilnya

# Tampilkan hasilnya ke layar
df_hasil.show()

Data Transaksi:
+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------+---------+-----------------+----------+
only showing top 5 rows

Data Pelanggan:
+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silve

**Soal 3.** Tulis ulang hasil gabungan Soal 2 (sebelum di-`groupBy`) ke disk lokal dalam format Parquet, dipartisi berdasarkan `membership`. Tampilkan struktur foldernya dengan `!ls`.

In [6]:
# Simpan DataFrame gabungan (sebelum groupBy) ke disk lokal dengan format Parquet dan dipartisi berdasarkan kolom 'membership'
df_gabung.write.mode("overwrite").partitionBy("membership").parquet("hasil_transaksi_parquet")

!ls hasil_transaksi_parquet

'membership=Gold'  'membership=Platinum'  'membership=Silver'   _SUCCESS


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa partitioning berdasarkan kolom `kota` cocok untuk skenario kita, namun akan menjadi **pilihan buruk** jika diterapkan pada kolom seperti `order_id` yang nilainya selalu unik di setiap baris? Tulis jawaban pada markdown cell di bawah ini.

Partisi berdasarkan kolom kota sangat cocok karena jumlah nilainya terbatas dan mengelompokkan transaksi ke dalam beberapa wilayah besar, sehingga mempercepat proses pencarian data (query pruning). Sebaliknya, menggunakan kolom order_id yang unik di setiap baris adalah pilihan buruk karena akan menciptakan jutaan folder kecil (disebut masalah small files problem). Hal ini akan membebani NameNode HDFS atau sistem file lokal dalam mengelola metadata, yang justru menurunkan performa Spark secara drastis.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [7]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

In [8]:
# Membaca data Transaksi (CSV)
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# Membaca data Produk (JSON)
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data Ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan Jumlah Baris
print(f"Jumlah baris df_transaksi : {df_transaksi.count()}")
print(f"Jumlah baris df_produk    : {df_produk.count()}")
print(f"Jumlah baris df_ulasan    : {df_ulasan.count()}\n")

# Menampilkan Schema masing-masing DataFrame
print("=== Schema Transaksi ===")
df_transaksi.printSchema()

print("=== Schema Produk ===")
df_produk.printSchema()

print("=== Schema Ulasan ===")
df_ulasan.printSchema()

Jumlah baris df_transaksi : 5000
Jumlah baris df_produk    : 30
Jumlah baris df_ulasan    : 3500

=== Schema Transaksi ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== Schema Produk ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== Schema Ulasan ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [9]:
# Join dari Transaksi ke Ulasan (Left Join karena tidak semua transaksi ada ulasan)
df_gabung_1 = df_transaksi.join(df_ulasan, on="order_id", how="left")

# Join ke master Produk (Inner Join karena produk pasti valid)
df_etl = df_gabung_1.join(df_produk, on="product_id", how="inner")

# Menambahkan kolom total_pendapatan (unit_terjual * harga)
df_etl = df_etl.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_etl.show()

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [10]:
from pyspark.sql.functions import when

# Tambahkan kolom ada_ulasan sebelum langkah na.fill()
df_etl = df_etl.withColumn(
    "ada_ulasan", 
    when(col("rating").isNotNull(), True).otherwise(False)
)

# Isi nilai kosong (null) pada kolom rating dengan angka 0
df_etl = df_etl.na.fill({"rating": 0})

df_cekNull = df_etl.filter(col("rating").isNull()).count()
print(f"Jumlah data rating yang NULL: {df_cekNull}")
df_etl.show()

Jumlah data rating yang NULL: 0
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|    

**Alasan Singkat Pengisian Nilai 0:**
Mengisi nilai null dengan angka 0 pada kolom rating sangat masuk akal karena angka 0 secara logis mempresentasikan kondisi "tidak ada nilai/skor yang diberikan" akibat pelanggan belum mengisi ulasan. Jika diisi dengan angka lain seperti 1 hingga 5, hal tersebut akan memalsukan atau merusak statistik penilaian asli dari produk yang memang sudah diulas secara aktif oleh pelanggan.

In [11]:
# Simpan hasil akhir ke HDFS format Parquet dipartisi berdasarkan 'kategori'
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs)

# Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Baca kembali data dari HDFS untuk memastikan integritas data
df_verifikasi = spark.read.parquet(path_hdfs)
print(f"\nBukti data tersimpan utuh. Jumlah baris terhitung kembali: {df_verifikasi.count()}")

-rw-r--r--   3 fais supergroup          0 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - fais supergroup          0 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 fais supergroup      16366 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-f7e91e46-034c-455e-9e3b-bb15d1ebe491.c000.snappy.parquet
drwxr-xr-x   - fais supergroup          0 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 fais supergroup      11133 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-f7e91e46-034c-455e-9e3b-bb15d1ebe491.c000.snappy.parquet
drwxr-xr-x   - fais supergroup          0 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 fais supergroup      10392 2026-10-26 04:01 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-f7e91e46-034c-455e-9e3b-bb15d1ebe491.c000.snappy.parquet
drwxr-xr-x   - fais supergroup 

In [12]:
from pyspark.sql.functions import count, avg, round

# Menghitung persentase transaksi dengan ulasan per kategori
df_insight = df_verifikasi.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    # Karena True bernilai 1 dan False bernilai 0 di tingkat sistem komputer saat dikalkulasi,
    # Kita bisa memanfaatkan rata-rata dari casting boolean ke integer untuk mencari persentase
    round(avg(col("ada_ulasan").cast("integer")) * 100, 2).alias("persentase_berulasan")
).orderBy("persentase_berulasan") # Urutkan dari terkecil ke terbesar

df_insight.show()

+------------+---------------+--------------------+
|    kategori|total_transaksi|persentase_berulasan|
+------------+---------------+--------------------+
|     Makanan|            536|               68.84|
|   Kesehatan|            961|               68.89|
|     Fashion|           1035|               70.14|
|Rumah Tangga|            815|               70.55|
|  Elektronik|           1653|               70.66|
+------------+---------------+--------------------+



**Interpretasi Bisnis**
Mengetahui kategori produk dengan persentase ulasan paling rendah sangat penting bagi tim marketing untuk mengidentifikasi adanya kendala post-purchase engagement pada kategori tersebut. Kategori yang minim ulasan dapat menurunkan tingkat kepercayaan calon pembeli baru, sehingga tim marketing perlu merancang strategi khusus seperti program insentif (misal: pemberian poin atau cashback) agar pelanggan lebih termotivasi memberikan ulasan setelah barang diterima.